In [ ]:
#!/usr/bin/env python3
"""
Simplified UpsilonTree |t| resolution and binning macro.
Optimized for CERN SWAN / Jupyter notebooks.
"""

import argparse
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import ROOT

METHODS = {
    "eX":     {"branch": "reco_t_eX",     "label": r"$|t|_{eX}$ lepton+$\Upsilon$"},
    "BABE":   {"branch": "reco_t_BABE",   "label": r"$|t|_{BABE}$ proton-only"},
    "eXBABE": {"branch": "reco_t_eXBABE", "label": r"$|t|_{eXBABE}$ hybrid"},
    "proj":   {"branch": "reco_t_proj",   "label": r"$|t|_{proj}$ projected/wedge", "weight": "weight_proj"},
}

def weighted_quantile(values, quantiles, sample_weight=None):
    """Calculates weighted quantiles."""
    values, quantiles = np.asarray(values), np.asarray(quantiles)
    if sample_weight is None:
        sample_weight = np.ones_like(values)
    
    # Sort and compute CDF
    sorter = np.argsort(values)
    values, sample_weight = values[sorter], np.asarray(sample_weight)[sorter]
    cdf = np.cumsum(sample_weight) - 0.5 * sample_weight
    total = np.sum(sample_weight)
    
    if total <= 0: return np.full(len(quantiles), np.nan)
    return np.interp(quantiles, cdf / total, values)

def purity_stability(H):
    """Calculates purity and stability from a 2D migration histogram (H)."""
    diag = np.diag(H)
    with np.errstate(divide="ignore", invalid="ignore"):
        purity = np.where(H.sum(axis=0) > 0, diag / H.sum(axis=0), np.nan)
        stability = np.where(H.sum(axis=1) > 0, diag / H.sum(axis=1), np.nan)
    return purity, stability

def plot_migration(H, edges, title, outpath):
    """Plots and saves the row-normalized migration matrix."""
    fig, ax = plt.subplots(figsize=(7, 6))
    
    # Row-normalize (Truth -> Reco)
    row_sums = H.sum(axis=1, keepdims=True)
    norm_H = np.divide(H, row_sums, out=np.zeros_like(H), where=row_sums > 0)
    
    im = ax.imshow(norm_H, origin="lower", aspect="auto", vmin=0, vmax=1)
    labels = [f"{edges[i]:.3g}-{edges[i+1]:.3g}" for i in range(len(edges)-1)]
    
    ax.set(xticks=np.arange(len(labels)), yticks=np.arange(len(labels)),
           xticklabels=labels, yticklabels=labels,
           xlabel=r"Reco $|t|$ bin [GeV$^2$]", ylabel=r"True $|t|$ bin [GeV$^2$]", title=title)
    ax.tick_params(axis='x', rotation=45)
    
    fig.colorbar(im, ax=ax, label="Fraction of truth bin")
    fig.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)

def main():
    ap = argparse.ArgumentParser()
    # Use flags with defaults so it runs smoothly in a notebook cell without args
    ap.add_argument("--input", default="~/merged_elecs_test_tuples.root", help="Input ROOT file")
    ap.add_argument("--tree", default="UpsilonTree", help="Name of the TTree inside the ROOT file")
    ap.add_argument("--outdir", default="~/t_binning_study")
    ap.add_argument("--edges", default="0,0.02,0.05,0.08,0.12,0.18,0.25,0.35,0.50,0.75,1.0,1.5,2.0")
    ap.add_argument("--methods", default="eX,BABE,eXBABE,proj")
    
    # The Jupyter/SWAN fix
    if "ipykernel" in sys.modules:
        args = ap.parse_args([]) # Ignore hidden '-f' flag in notebooks
    else:
        args = ap.parse_args()

    os.makedirs(args.outdir, exist_ok=True)
    edges = np.array([float(x) for x in args.edges.split(",")])
    methods = [m.strip() for m in args.methods.split(",")]

    # 1. Load Data with RDataFrame and convert straight to Pandas
    base_cut = "true_t > 0 && reco_mass >= 8.0 && reco_mass <= 12.0 && reco_W >= 20.0"
    rdf = ROOT.RDataFrame(args.tree, args.input).Filter(base_cut)
    
    # Grab only the columns we actually need to save memory
    req_cols = ["true_t", "weight", "weight_proj"] + [METHODS[m]["branch"] for m in methods if m in METHODS]
    df = pd.DataFrame(rdf.AsNumpy(columns=req_cols))

    summary_data = []

    # 2. Process each method
    for method in methods:
        if method not in METHODS:
            continue
            
        info = METHODS[method]
        reco_col = info["branch"]
        weight_col = info.get("weight", "weight")

        # Filter out invalid events for this specific method
        m_df = df[(df[reco_col] > 0) & (df[weight_col] > 0)].copy()
        
        if m_df.empty:
            print(f"Skipping {method} - no valid data found.")
            continue
            
        true_t, reco_t, w = m_df["true_t"], m_df[reco_col], m_df[weight_col]

        # NumPy's built-in 2D histogram builds the migration matrix instantly
        H, _, _ = np.histogram2d(true_t, reco_t, bins=edges, weights=w)
        pur, stab = purity_stability(H)

        # Plot Migration
        plot_migration(H, edges, f"Migration: {info['label']}", os.path.join(args.outdir, f"migration_{method}.png"))

        # Calculate Resolution iteratively per bin
        m_df["bin"] = pd.cut(m_df["true_t"], bins=edges, labels=False)
        for i in range(len(edges) - 1):
            bin_data = m_df[m_df["bin"] == i]
            n_events = len(bin_data)
            sum_w = bin_data[weight_col].sum() if n_events > 0 else 0

            row = {
                "method": method, "t_low": edges[i], "t_high": edges[i+1],
                "n": n_events, "sumw": sum_w, 
                "purity": pur[i] if i < len(pur) else np.nan, 
                "stability": stab[i] if i < len(stab) else np.nan
            }

            if n_events > 0:
                dt = bin_data[reco_col] - bin_data["true_t"]
                rel = dt / bin_data["true_t"]
                q_rel = weighted_quantile(rel, [0.16, 0.50, 0.84], bin_data[weight_col])
                row.update({"median_rel": q_rel[1], "core_rel": 0.5 * (q_rel[2] - q_rel[0])})
            else:
                row.update({"median_rel": np.nan, "core_rel": np.nan})
            
            summary_data.append(row)

    # 3. Save Summary to CSV easily using Pandas
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        out_csv = os.path.join(args.outdir, "t_resolution_summary.csv")
        summary_df.to_csv(out_csv, index=False)
        print(f"Done! Plots and CSVs saved to: {args.outdir}/")
    else:
        print("No valid data processed. CSV not written.")

if __name__ == "__main__":
    main()